In [1]:
import torch
from IPython.display import display

from synthetic_data_generation import make_cohort
import pandas as pd

RANDOM_STATE = 0

# Generate Synthetic Data

In [2]:
data_folder = '../../../data/synthetic_data'
make_cohort(data_folder)

In [3]:
tabular_df = pd.read_csv(data_folder + '/clinical.csv', index_col=0)
display(tabular_df.head())

,age,stage,gender,time,event
patient_id,,,,,
patient_000,63.047171,NaN,NaN,166.605893,True
patient_001,49.600159,2.0,F,2020.062176,False
patient_002,67.504512,4.0,F,43.528422,True
patient_003,69.405647,4.0,M,43.353612,True
patient_004,40.489648,2.0,F,711.328066,True


# Data Loading

## Tabular

In [4]:
from kalecancer.loaddata import Tabular

clinical_df = tabular_df.drop(columns =['time', 'event'])

clinical_modality = Tabular(clinical_df)

# proof it works!
display(clinical_modality.frame.head())
display(clinical_modality['patient_000'])
display(clinical_modality.ids[:5])
display((clinical_modality.frame == clinical_df).any().any())
display((clinical_modality.ids == clinical_modality.frame.index).any())
display((clinical_df.index == clinical_modality.ids).any())

,age,stage,gender
id,,,
patient_000,63.047171,NaN,NaN
patient_001,49.600159,2.0,F
patient_002,67.504512,4.0,F
patient_003,69.405647,4.0,M
patient_004,40.489648,2.0,F


,age,stage,gender
id,,,
patient_000,63.047171,NaN,NaN


Index(['patient_000', 'patient_001', 'patient_002', 'patient_003',
       'patient_004'],
      dtype='object', name='id')

np.True_

np.True_

np.True_


## Patch Features
### Whole Slide Images passed through an encoder such as UNI



In [5]:
# What does the file structure look like
from pathlib import Path
list(Path(data_folder + '/patches').glob('*.h5'))[:5]

[PosixPath('../../../data/synthetic_data/patches/patient_027.h5'),
 PosixPath('../../../data/synthetic_data/patches/patient_082.h5'),
 PosixPath('../../../data/synthetic_data/patches/patient_057.h5'),
 PosixPath('../../../data/synthetic_data/patches/patient_044.h5'),
 PosixPath('../../../data/synthetic_data/patches/patient_026.h5')]

In [6]:
from kalecancer.loaddata import PatchFeatures

patch_features = PatchFeatures.from_glob(

    # 'what files are the patches stored in?'
    pattern = data_folder + '/patches/*.h5',

    # 'how to extract the ids from these files'
    id_pattern= r'(patient_\d{3})\.h5'
)

display(patch_features.ids[:5])
display(patch_features['patient_000'].shape)
display(patch_features['patient_000'].head())

Index(['patient_000', 'patient_001', 'patient_002', 'patient_003',
       'patient_004'],
      dtype='object', name='id')

(168, 35)

file      x  \
id          patch                                                             
patient_000 0      ../../../data/synthetic_data/patches/patient_0...  26803   
            1      ../../../data/synthetic_data/patches/patient_0...  25438   
            2      ../../../data/synthetic_data/patches/patient_0...  36345   
            3      ../../../data/synthetic_data/patches/patient_0...  35263   
            4      ../../../data/synthetic_data/patches/patient_0...  41517   

                       y  feature_0  feature_1  feature_2  feature_3  \
id          patch                                                      
patient_000 0      30774   3.152563   1.471492  -2.566658  -0.236850   
            1      16956   3.750869   1.820646   0.730775  -1.572040   
            2      43944   3.232878   0.234401   0.270343  -0.863345   
            3      38608   3.524188   0.276274  -1.412766  -2.310103   
            4      18368   4.353586   0.092127  -0.837398  -0.594400   

                   feature_4  feature_5  feature_6  ...  feature_22  \
id          patch                                   ...               
patient_000 0       0.176512   0.295994  -0.371915  ...    0.438993   
            1      -0.066953  -1.172007  -0.518280  ...   -2.063238   
            2      -0.147529  -0.152523   0.383394  ...   -2.003522   
            3       0.054354  -0.471776   0.459386  ...   -1.296472   
            4      -1.480536  -0.888134  -0.358017  ...    0.328662   

                   feature_23  feature_24  feature_25  feature_26  feature_27  \
id          patch                                                               
patient_000 0        0.827988   -0.296571    0.066546   -0.697424    0.989584   
            1       -0.591103    0.590906   -1.581594    1.475949    0.368357   
            2        1.604254   -0.457699    0.107880    1.309551   -1.602260   
            3       -1.482185   -2.333616   -0.678264    0.749434   -0.284884   
            4        0.361922    1.320662   -0.342786   -1.476858    1.067222   

                   feature_28  feature_29  feature_30  feature_31  
id          patch                                                  
patient_000 0       -1.178304    0.782350   -0.190651    1.171247  
            1        0.846584   -0.570944    0.813764    1.068472  
            2       -1.251647   -1.601278   -0.794136    0.439637  
            3        0.197790    1.089218    1.327686   -0.069138  
            4       -0.331488    1.114592    0.383377   -0.131138  

[5 rows x 35 columns]

## The Target

In [7]:
target_df = tabular_df[['time', 'event']]
display(target_df.head(5))

,time,event
patient_id,,
patient_000,166.605893,True
patient_001,2020.062176,False
patient_002,43.528422,True
patient_003,43.353612,True
patient_004,711.328066,True


In [8]:
from kalecancer.loaddata import TimeToEvent

target = TimeToEvent(time = target_df['time'], event=target_df['event'])
display(target.ids[:5])
display(target[['patient_000', 'patient_003']])


Index(['patient_000', 'patient_001', 'patient_002', 'patient_003',
       'patient_004'],
      dtype='object', name='id')

,time,event
id,,
patient_000,166.605896,True
patient_003,43.353611,True


## Multimodal Dataset: All Modalities Combined

In [9]:
from kalecancer.loaddata import MultimodalDataset

dataset = MultimodalDataset(
    modalities={
       'clinical': clinical_modality,
       'wsi': patch_features
    },
    target=target,
    required_modalities=['clinical', 'wsi']
)

display(dataset.ids[:5])
display(dataset.summary())
display(dataset['patient_052'].keys())
display(dataset['patient_052']['clinical'])

Index(['patient_000', 'patient_001', 'patient_002', 'patient_003',
       'patient_004'],
      dtype='object', name='id')

,count
available: clinical,100
available: wsi,100
available: target,100
included,100
included with clinical,100
included with wsi,100
events,90


dict_keys(['clinical', 'wsi', 'target'])

,age,stage,gender
id,,,
patient_052,45.428442,3.0,F


# Model Building

## Intermediate Fusion
```mermaid
flowchart LR
    A[Modality A<br/>e.g. Clinical table] --> EA[Encoder A]
    B[Modality B<br/>e.g. Pathology patches] --> EB[Encoder B]
    EA -->|embedding A| F((Fusion))
    EB -->|embedding B| F
    F -->|fused representation| H[Shared prediction head]
    H --> O([Prediction])
```


In [10]:
# Needed to ensure model weights are initialised consistently
torch.manual_seed(RANDOM_STATE)

In [11]:
dataset.modalities.keys()

dict_keys(['clinical', 'wsi'])

In [12]:
dataset.modalities['clinical'].frame.shape

(100, 3)

In [13]:
dataset['patient_000']['wsi'].columns

Index(['file', 'x', 'y', 'feature_0', 'feature_1', 'feature_2', 'feature_3',
       'feature_4', 'feature_5', 'feature_6', 'feature_7', 'feature_8',
       'feature_9', 'feature_10', 'feature_11', 'feature_12', 'feature_13',
       'feature_14', 'feature_15', 'feature_16', 'feature_17', 'feature_18',
       'feature_19', 'feature_20', 'feature_21', 'feature_22', 'feature_23',
       'feature_24', 'feature_25', 'feature_26', 'feature_27', 'feature_28',
       'feature_29', 'feature_30', 'feature_31'],
      dtype='object')

In [14]:
from kalecancer.model import ABMIL, IntermediateFusion, Concat, CoxHead

inter_model = IntermediateFusion(
    encoders = {
        'clinical': [torch.nn.Linear(dataset.modalities['clinical'].frame.shape[1], 8)],
        'wsi': [
            ABMIL(32, hidden_dim=16, attention_dim=8, dropout=0.0),
            torch.nn.Linear(16,8)
        ]
    },

    # the encoder outputs concatenated
    fusion = Concat(),

    head = CoxHead(in_dim=16, ties='efron')
)

In [15]:
display(inter_model.modalities)
display(inter_model.encoders['clinical'])
display(inter_model.encoders['wsi'])
display(inter_model.fusion)
display(inter_model.head)

['clinical', 'wsi']

EncoderList(
  (0): Linear(in_features=3, out_features=8, bias=True)
)

EncoderList(
  (0): ABMIL(
    (embed): Sequential(
      (0): Linear(in_features=32, out_features=16, bias=True)
      (1): ReLU()
      (2): Dropout(p=0.0, inplace=False)
    )
    (attention_value): Sequential(
      (0): Linear(in_features=16, out_features=8, bias=True)
      (1): Tanh()
    )
    (attention_gate): Sequential(
      (0): Linear(in_features=16, out_features=8, bias=True)
      (1): Sigmoid()
    )
    (attention_score): Linear(in_features=8, out_features=1, bias=True)
  )
  (1): Linear(in_features=16, out_features=8, bias=True)
)

Concat()

CoxHead(
  (linear): Linear(in_features=16, out_features=1, bias=False)
)

# The Model Orchestrator???? - naming needs some work

In [16]:
# getting insight into the tabular data, needed to understand what transforms to apply
print('Number of missing values:')
display(dataset.modalities['clinical'].frame.isna().sum())
print('\nNumber unique values:')
display(dataset.modalities['clinical'].frame.nunique())

Number of missing values:


age       13
stage     13
gender     7
dtype: int64


Number unique values:


age       87
stage      4
gender     2
dtype: int64

In [17]:
dataset['patient_002']['clinical']

,age,stage,gender
id,,,
patient_002,67.504512,4.0,F


In [18]:
# from the above information we make must make the following transforms.
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline

numeric_pipeline = make_pipeline(SimpleImputer(), StandardScaler())

tab_transfromers = ColumnTransformer([
    ('numeric', numeric_pipeline, ['age']),
    ('categorical', OrdinalEncoder(encoded_missing_value=-2), ['stage', 'gender'])
])

In [19]:
from kalecancer.pipeline import Pipeline
from functools import partial

pipeline = Pipeline(
    model= inter_model,

    # these are transforms that will be applied to the dataset
    # currently only Tabular modalities accept transforms
    # transforms need to ouptut numerical values, and their output should be
    # accepted by the first encoder
    transforms={'clinical': tab_transfromers},

    optimizer= partial(torch.optim.AdamW, lr=1e-3),
    batch_size=32,
    drop_last=True,
    max_epochs=40,
    validation=None,
    early_stopping=None,
    accelerator='cpu',
    precision='32-true',
    random_state=RANDOM_STATE,
)

# Training

In [20]:
from kalecancer.loaddata import train_test_split
train, test = train_test_split(dataset, test_size=0.2, stratify=True, random_state=RANDOM_STATE)

display(len(train))
display(len(test))

80

20

In [21]:
pipeline.fit(train)

,model,IntermediateF...s=False) ) )
,transforms,"{'clinical': ColumnTransfo..., 'gender'])])}"
,optimizer,"functools.par...W'>, lr=0.001)"
,batch_size,32
,drop_last,True
,max_epochs,40
,validation,None
,early_stopping,None
,accelerator,'cpu'
,precision,'32-true'
,random_state,0


In [22]:
# [OPTIONAL]
# Insight into how the transformers are applied in the Pipeline.fit

fitted_t = tab_transfromers.fit(train.modalities['clinical'].frame)
fitted_train = train.modalities['clinical'].with_transform(fitted_t, train.ids)

print('Number of missing values before transforms:')
print(train.modalities['clinical'].frame.isna().sum().sum())

print()
print('Number of missing values after transforms:')
print(fitted_train.frame.isna().sum().sum())


Number of missing values before transforms:
33

Number of missing values after transforms:
0


In [23]:
display(pipeline.history_.head())

,train_loss,optimizer_steps,skipped_batches
epoch,,,
0,2.730898,2,0
1,2.746190,4,0
2,2.596660,6,0
3,2.726141,8,0
4,2.699171,10,0


# Evaluation
Note that we are using small synthetic illustrative cohort, so results don't provide real clinical insight

In [24]:
pipeline.predict(test).head()

,log_hazard
id,
patient_001,-0.698043
patient_010,0.089252
patient_013,-0.010415
patient_021,-0.372110
patient_027,0.088047


In [25]:
from kalecancer.evaluate import HarrellC, UnoC
pipeline.evaluate(test, metrics={"harrell_c": HarrellC(), "uno_c_5y": UnoC(tau=1825.0)})

harrell_c    0.629032
uno_c_5y     0.617628
Name: score, dtype: float64

## [Advanced] Comparing Models

In [26]:
from kalecancer.model import Unimodal
from sklearn.base import clone
torch.manual_seed(0)
clinical_model = Unimodal(
    "clinical",
    encoders=[torch.nn.Linear(dataset.modalities['clinical'].frame.shape[1], 8)],
    head=CoxHead(in_dim=8, ties="efron"),
)
baseline_pipeline = clone(pipeline).set_params(model=clinical_model).fit(train)

In [27]:
from kalecancer.evaluate import EvalContext, bootstrap_ci
prediction, baseline_prediction = pipeline.predict(test), baseline_pipeline.predict(test)

target = test.target.frame.loc[prediction.index]
context = EvalContext(train_target=pipeline.train_target_, classes=None)
baseline_context = EvalContext(train_target=baseline_pipeline.train_target_, classes=None)

intervals = {
    "fused": bootstrap_ci(HarrellC(), prediction, target, context),
    "clinical only": bootstrap_ci(HarrellC(), baseline_prediction, target, baseline_context),
    "gain from the slides": bootstrap_ci(
        HarrellC(), prediction, target, context, baseline=baseline_prediction, baseline_context=baseline_context
    ),
}
table = pd.DataFrame({name: vars(ci) for name, ci in intervals.items()}).T[["estimate", "lower", "upper"]]
# explicit, as not every editor renders a bare DataFrame on the last line (PyCharm, for one)
display(table)

,estimate,lower,upper
fused,0.629032,0.483324,0.753042
clinical only,0.607527,0.445618,0.765802
gain from the slides,0.021505,-0.060797,0.104651


In [28]:
# required to solve display() rendering issues in prior cells
display(pd.DataFrame(data={}, index=[i for i in range(100)]))

""
0
1
2
3
4
...
95
96
97
98
